In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)

X_train = pd.read_csv("../data/processed/X_train_clip.csv")
X_test = pd.read_csv("../data/processed/X_test_clip.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

# Why each score is there

## Accuracy:
 To show you the trap.
With 4.7% bankrupt, a lazy model that says “healthy” for everyone gets about 95%. We calculate it so you can see it looks great while being useless. It’s a warning sign, not a goal.

## Recall: 
To count the missed bankruptcies.
“Out of 100 truly bankrupt companies, how many did we catch?” If recall is 0.70, we caught 70 and missed 30. For a bank, this is the most important number, because misses cost the most.

## Precision:
 To count the false alarms.
“When the model shouts bankrupt, how often is it right?” If we catch every bankruptcy by labeling everyone bankrupt, recall is perfect but the model is useless. Precision stops us from cheating that way.

## F1: 
To give one fair number for the tug-of-war.
Recall and precision pull against each other. Catch more bankruptcies and you get more false alarms. F1 blends them into one number, so it’s easy to compare many models in a table.

## ROC-AUC: 
To judge the model without picking a cutoff.
The model really gives a chance from 0% to 100%, and we choose the cutoff (like 50%) that turns it into “bankrupt” or “healthy”. ROC-AUC checks if the model ranks risky companies above safe ones no matter where we draw the line. It’s useful for fairly comparing models, and later for tuning the cutoff.

In [2]:
# Scale: learn from train, apply to both
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [3]:
# A small helper so we can score every model the same way
def score_model(name, model, X_te, y_te):
    pred = model.predict(X_te)
    proba = model.predict_proba(X_te)[:, 1]
    print(f"{name}")
    print("  Accuracy :", round(accuracy_score(y_te, pred), 3))
    print("  Precision:", round(precision_score(y_te, pred, zero_division=0), 3))
    print("  Recall   :", round(recall_score(y_te, pred), 3))
    print("  F1       :", round(f1_score(y_te, pred), 3))
    print("  ROC-AUC  :", round(roc_auc_score(y_te, proba), 3))

In [4]:
# 1. The lazy model: always says "healthy"
lazy = DummyClassifier(strategy="most_frequent")
lazy.fit(X_train_s, y_train)
score_model("Lazy model", lazy, X_test_s, y_test)

Lazy model
  Accuracy : 0.953
  Precision: 0.0
  Recall   : 0.0
  F1       : 0.0
  ROC-AUC  : 0.5


In [5]:
# 2. Real Logistic Regression
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train_s, y_train)
score_model("Logistic Regression", logreg, X_test_s, y_test)

Logistic Regression
  Accuracy : 0.953
  Precision: 0.6
  Recall   : 0.03
  F1       : 0.058
  ROC-AUC  : 0.75


In [6]:
# 3. Same model, but told to care more about the rare bankrupt companies
logreg_bal = LogisticRegression(max_iter=1000, class_weight="balanced")
logreg_bal.fit(X_train_s, y_train)
score_model("Logistic Regression (balanced)", logreg_bal, X_test_s, y_test)

Logistic Regression (balanced)
  Accuracy : 0.713
  Precision: 0.104
  Recall   : 0.667
  F1       : 0.179
  ROC-AUC  : 0.741
